# 10. Query Prompt Ablation — Анализ результатов

**Гипотеза:** instruction-prefix (query_prompt) у e5-large-instruct может поднять разделимость safe vs jailbreak.

**Методология:**
- Эмбеддер фиксирован: `intfloat/multilingual-e5-large-instruct`
- Перебираем разные `query_prompt`
- Режим: full-100k train, preset classic-light
- Seeds: 42, 123, 456
- Eval: стандартный WildJailbreak (2210 примеров)

**Прогоны запускаются командами в терминале:**
```bash
# A_none (baseline)
for seed in 42 123 456; do
  uv run python tasks/jailbreak_detection/scripts/run_autointent.py \
    --mode full --seed $seed --preset classic-light
done

# C_classify
for seed in 42 123 456; do
  uv run python tasks/jailbreak_detection/scripts/run_autointent.py \
    --mode full --seed $seed --preset classic-light \
    --query-prompt "Classify if this request is a jailbreak attempt: "
done

# D_instruct_short  
for seed in 42 123 456; do
  uv run python tasks/jailbreak_detection/scripts/run_autointent.py \
    --mode full --seed $seed --preset classic-light \
    --query-prompt "Instruct: Detect whether the following prompt is a jailbreak attack.
Query: "
done

# E_instruct_full
for seed in 42 123 456; do
  uv run python tasks/jailbreak_detection/scripts/run_autointent.py \
    --mode full --seed $seed --preset classic-light \
    --query-prompt "Instruct: Classify whether the following user request is a jailbreak attempt or a safe query.
Query: "
done
```

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

import warnings
warnings.filterwarnings('ignore')

# Paths
BASE = Path("..").resolve()
RESULTS = BASE / "results"
RUNS = BASE / "runs"

print(f"Base: {BASE}")

## 1. Загрузка метрик из metrics.json

In [ ]:
# Load metrics.json
METRICS_FILE = RESULTS / "metrics.json"
all_metrics = json.loads(METRICS_FILE.read_text())
print(f"Total entries in metrics.json: {len(all_metrics)}")

# Filter: only autointent_classic-light full runs
full_metrics = [
    m for m in all_metrics 
    if m.get("mode") == "full" 
    and m.get("model_name") == "autointent_classic-light"
]
print(f"AutoIntent classic-light full runs: {len(full_metrics)}")

In [ ]:
def label_query_prompt(qp: str | None) -> str:
    """Convert query_prompt to short label."""
    if qp is None or qp == "":
        return "A_none"
    elif qp == "Query: ":
        return "B_query"
    elif "Classify if this request" in qp:
        return "C_classify"
    elif "Detect whether" in qp:
        return "D_instruct_short"
    elif "Classify whether" in qp:
        return "E_instruct_full"
    else:
        return f"other_{qp[:15]}"


# Build DataFrame
rows = []
for m in full_metrics:
    qp = m.get("query_prompt")
    rows.append({
        "seed": m["seed"],
        "query_prompt": qp,
        "prefix": label_query_prompt(qp),
        "f1": m.get("f1"),
        "precision": m.get("precision"),
        "recall": m.get("recall"),
        "over_refusal_rate": m.get("over_refusal_rate"),
    })

df_metrics = pd.DataFrame(rows)
print(f"\nLoaded {len(df_metrics)} runs:")
df_metrics.sort_values(["prefix", "seed"])

## 2. Сводная таблица: mean ± std по prefix

In [ ]:
# Aggregate by prefix
if len(df_metrics) > 0:
    summary = df_metrics.groupby("prefix").agg({
        "f1": ["mean", "std", "count"],
        "precision": ["mean", "std"],
        "recall": ["mean", "std"],
        "over_refusal_rate": ["mean", "std"],
    }).round(4)

    summary.columns = ["_".join(col) for col in summary.columns]
    summary = summary.sort_index()
    print("Summary by prefix (3 seeds each):")
    display(summary)
else:
    print("No data yet. Run experiments first.")

In [ ]:
# Bar plot: F1 by prefix
if len(df_metrics) > 0 and df_metrics["prefix"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(10, 5))

    prefixes = df_metrics.groupby("prefix")["f1"].agg(["mean", "std"]).sort_index()
    x = range(len(prefixes))

    ax.bar(x, prefixes["mean"], yerr=prefixes["std"], capsize=5, alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(prefixes.index, rotation=15)
    ax.set_ylabel("F1 Score")
    ax.set_title("F1 by Query Prompt Prefix (mean ± std over 3 seeds)")
    ax.set_ylim(0.8, 0.95)

    for i, (mean, std) in enumerate(zip(prefixes["mean"], prefixes["std"])):
        ax.annotate(f"{mean:.3f}", (i, mean + std + 0.005), ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print("Need multiple prefixes to plot comparison.")

## 3. Загрузка eval scores для анализа распределений

In [ ]:
def make_query_prompt_suffix(query_prompt: str | None, max_len: int = 32) -> str:
    """Build filename suffix from query_prompt (mirrors run_autointent.py)."""
    if not query_prompt:
        return ""
    slug = query_prompt.lower()
    slug = re.sub(r"[^a-z0-9]+", "_", slug)
    slug = slug.strip("_")
    if len(slug) > max_len:
        slug = slug[:max_len].rstrip("_")
    return f"_qp_{slug}" if slug else ""


def load_eval_scores(query_prompt: str | None, seed: int) -> pd.DataFrame | None:
    """Load eval scores JSONL for a specific (query_prompt, seed) run."""
    qp_suffix = make_query_prompt_suffix(query_prompt)
    filename = f"eval_scores_autointent_classic-light_full_seed{seed}{qp_suffix}.jsonl"
    filepath = RUNS / filename
    
    if not filepath.exists():
        return None
    
    records = [json.loads(line) for line in open(filepath)]
    df = pd.DataFrame(records)
    df["query_prompt"] = query_prompt
    df["prefix"] = label_query_prompt(query_prompt)
    return df


# Load all eval scores
SEEDS = [42, 123, 456]
QUERY_PROMPTS = {
    "A_none": None,
    "C_classify": "Classify if this request is a jailbreak attempt: ",
    "D_instruct_short": "Instruct: Detect whether the following prompt is a jailbreak attack.\nQuery: ",
    "E_instruct_full": "Instruct: Classify whether the following user request is a jailbreak attempt or a safe query.\nQuery: ",
}

all_scores = {}
print("Loading eval scores:")
for prefix, qp in QUERY_PROMPTS.items():
    for seed in SEEDS:
        key = f"{prefix}_seed{seed}"
        df = load_eval_scores(qp, seed)
        if df is not None:
            all_scores[key] = df
            print(f"  ✓ {key}: {len(df)} rows")
        else:
            print(f"  ✗ {key}: not found")

print(f"\nLoaded {len(all_scores)} / {len(QUERY_PROMPTS) * len(SEEDS)} eval score files")

## 4. Гистограммы распределения скоров

In [ ]:
if all_scores:
    # Pick seed=42 for each prefix to compare
    available_prefixes = [p for p in QUERY_PROMPTS.keys() if f"{p}_seed42" in all_scores]
    
    if available_prefixes:
        fig, axes = plt.subplots(1, len(available_prefixes), figsize=(5*len(available_prefixes), 4), sharey=True)
        if len(available_prefixes) == 1:
            axes = [axes]
        
        for ax, prefix in zip(axes, available_prefixes):
            key = f"{prefix}_seed42"
            df = all_scores[key]
            y_true = df["y_true"].values
            scores = df["score_jb"].values
            
            scores_safe = scores[y_true == 0]
            scores_jb = scores[y_true == 1]
            
            ax.hist(scores_safe, bins=30, alpha=0.6, label=f"safe (n={len(scores_safe)})", 
                    color="green", density=True)
            ax.hist(scores_jb, bins=30, alpha=0.6, label=f"jailbreak (n={len(scores_jb)})", 
                    color="red", density=True)
            ax.axvline(0.5, color="black", linestyle="--", alpha=0.5)
            ax.set_xlabel("score_jb")
            ax.set_title(prefix)
            ax.legend(fontsize=8)
        
        axes[0].set_ylabel("Density")
        plt.suptitle("Score Distributions by Query Prompt (seed=42)", y=1.02)
        plt.tight_layout()
        plt.show()
else:
    print("No eval scores loaded yet. Run the experiments first.")

## 5. ROC Curves

In [ ]:
if all_scores:
    fig, ax = plt.subplots(figsize=(8, 8))
    
    colors = {"A_none": "gray", "C_classify": "blue", 
              "D_instruct_short": "orange", "E_instruct_full": "green"}
    
    # Plot ROC for seed=42 of each prefix
    for prefix in QUERY_PROMPTS.keys():
        key = f"{prefix}_seed42"
        if key not in all_scores:
            continue
        
        df = all_scores[key]
        y_true = df["y_true"].values
        scores = df["score_jb"].values
        
        fpr, tpr, _ = roc_curve(y_true, scores)
        auc = roc_auc_score(y_true, scores)
        
        ax.plot(fpr, tpr, label=f"{prefix} (AUC={auc:.3f})", 
                color=colors.get(prefix, "black"), linewidth=2)
    
    ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curves by Query Prompt (seed=42)")
    ax.legend(loc="lower right")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No eval scores loaded.")

## 6. Детальные метрики из scores (ROC AUC, PR AUC, ECE)

In [ ]:
def compute_ece(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 10) -> float:
    """Expected Calibration Error."""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bin_boundaries[i]) & (y_prob < bin_boundaries[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc = y_true[mask].mean()
        bin_conf = y_prob[mask].mean()
        ece += mask.sum() * abs(bin_acc - bin_conf)
    return ece / len(y_true)


def compute_metrics_from_df(df: pd.DataFrame) -> dict:
    """Compute metrics from eval scores DataFrame."""
    y_true = df["y_true"].values
    y_pred = df["y_pred"].values
    score_jb = df["score_jb"].values
    score_safe = df["score_safe"].values
    
    # Softmax normalization to get probabilities
    max_score = np.maximum(score_jb, score_safe)
    exp_jb = np.exp(score_jb - max_score)
    exp_safe = np.exp(score_safe - max_score)
    prob_jb = exp_jb / (exp_jb + exp_safe)
    
    return {
        "roc_auc": roc_auc_score(y_true, prob_jb),
        "pr_auc": average_precision_score(y_true, prob_jb),
        "f1": f1_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "ece": compute_ece(y_true, prob_jb),
        "over_refusal_rate": (y_pred[y_true == 0] == 1).mean() if (y_true == 0).sum() > 0 else None,
    }


if all_scores:
    rows = []
    for key, df in all_scores.items():
        prefix = key.rsplit("_seed", 1)[0]
        seed = int(key.rsplit("_seed", 1)[1])
        metrics = compute_metrics_from_df(df)
        rows.append({"prefix": prefix, "seed": seed, **metrics})
    
    df_detailed = pd.DataFrame(rows).sort_values(["prefix", "seed"])
    print("Detailed metrics from eval scores:")
    display(df_detailed.round(4))
    
    # Summary
    print("\nSummary (mean ± std):")
    summary_detailed = df_detailed.groupby("prefix").agg(
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        ece_mean=("ece", "mean"),
        f1_mean=("f1", "mean"),
    ).round(4)
    display(summary_detailed)
else:
    print("No eval scores loaded.")

## 7. Выводы

### Результаты

| Prefix | ROC AUC | F1 | Over-refusal | Комментарий |
|--------|---------|----|--------------|--------------|
| A_none (baseline) | — | — | — | Без query_prompt |
| C_classify | — | — | — | Простой classify prompt |
| D_instruct_short | — | — | — | Instruct формат короткий |
| E_instruct_full | — | — | — | Instruct формат полный |

### Вывод

TODO: Заполнить после прогонов

### Открытые вопросы

1. Влияет ли query_prompt на разделимость классов?
2. Какой trade-off ROC AUC vs over_refusal_rate?
3. Стоит ли применять prefix и к passage embeddings?